## Data Profiling

In [1]:
import pandas as pd
import sqlalchemy as sq
import pymysql as psql

print(sq.__version__)
print(pd.__version__)


2.0.49
3.0.2


In [4]:
from sqlalchemy import *
from pandas import *
from pymysql import *

from dotenv import load_dotenv
import os

load_dotenv()

user     = os.getenv("DB_USER")
pwd      = os.getenv("DB_PWD")
host     = os.getenv("DB_HOST")
port     = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

engine = create_engine(f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{database}")


In [6]:
import pandas as pd

df = pd.read_sql("SELECT * FROM staging_transactions", engine)

print(df.shape)
print(df.dtypes)

(2512, 16)
transaction_id                          str
account_id                              str
transaction_amount                  float64
transaction_date             datetime64[us]
transaction_type                        str
location                                str
device_id                               str
ip_address                              str
merchant_id                             str
channel                                 str
customer_age                          int64
customer_occupation                     str
transaction_duration                  int64
login_attempts                        int64
account_balance                     float64
previous_transaction_date    datetime64[us]
dtype: object


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   transaction_id             2512 non-null   str           
 1   account_id                 2512 non-null   str           
 2   transaction_amount         2512 non-null   float64       
 3   transaction_date           2512 non-null   datetime64[us]
 4   transaction_type           2512 non-null   str           
 5   location                   2512 non-null   str           
 6   device_id                  2512 non-null   str           
 7   ip_address                 2512 non-null   str           
 8   merchant_id                2512 non-null   str           
 9   channel                    2512 non-null   str           
 10  customer_age               2512 non-null   int64         
 11  customer_occupation        2512 non-null   str           
 12  transaction_durat

In [ ]:
df.describe()

,transaction_amount,transaction_date,customer_age,transaction_duration,login_attempts,account_balance,previous_transaction_date
count,2512.000000,2512,2512.000000,2512.000000,2512.000000,2512.000000,2512
mean,297.593778,2023-07-05 20:32:10.826433,44.673965,119.643312,1.124602,5114.302966,2024-11-04 08:09:22.219745
min,0.260000,2023-01-02 16:00:06,18.000000,10.000000,1.000000,101.250000,2024-11-04 08:06:23
25%,81.885000,2023-04-03 16:22:05.750000,27.000000,63.000000,1.000000,1504.370000,2024-11-04 08:07:53
50%,211.140000,2023-07-07 17:49:43.500000,45.000000,112.500000,1.000000,4735.510000,2024-11-04 08:09:22
75%,414.527500,2023-10-06 18:40:53.500000,59.000000,161.000000,1.000000,7678.820000,2024-11-04 08:10:53.250000
max,1919.110000,2024-01-01 18:21:50,80.000000,300.000000,5.000000,14977.990000,2024-11-04 08:12:23
std,291.946243,NaN,17.792198,69.963757,0.602662,3900.942499,NaN


#### Check 1: Finding Duplicates

In [14]:
dupe_count = df.duplicated(subset=['transaction_id']).sum()
print(f"Duplicate transaction_ids: {dupe_count}")

# Show the actual duplicate rows if any exist
dupes = df[df.duplicated(subset=['transaction_id'], keep=False)]
print(f"Rows involved in duplicates: {len(dupes)}")
print(dupes)


Duplicate transaction_ids: 0
Rows involved in duplicates: 0
Empty DataFrame
Columns: [transaction_id, account_id, transaction_amount, transaction_date, transaction_type, location, device_id, ip_address, merchant_id, channel, customer_age, customer_occupation, transaction_duration, login_attempts, account_balance, previous_transaction_date]
Index: []


#### Check 2 : Nulls and Empty Strings

In [16]:
null_counts = df.isnull().sum()
empty_counts = (df == '').sum()

profile = pd.DataFrame({
    'null_count': null_counts,
    'empty_count': empty_counts
})

print(profile)

                           null_count  empty_count
transaction_id                      0            0
account_id                          0            0
transaction_amount                  0            0
transaction_date                    0            0
transaction_type                    0            0
location                            0            0
device_id                           0            0
ip_address                          0            0
merchant_id                         0            0
channel                             0            0
customer_age                        0            0
customer_occupation                 0            0
transaction_duration                0            0
login_attempts                      0            0
account_balance                     0            0
previous_transaction_date           0            0


In [ ]:
cat_cols = ["transaction_type","channel","customer_occupation"]

for col in cat_cols:
    print(f"\n---{col}---")
    print(df[col].str.strip().unique())
    print(df[col].value_counts())



---transaction_type---
<StringArray>
['Debit', 'Credit']
Length: 2, dtype: str
transaction_type
Debit     1944
Credit     568
Name: count, dtype: int64

---channel---
<StringArray>
['ATM', 'Online', 'Branch']
Length: 3, dtype: str
channel
Branch    868
ATM       833
Online    811
Name: count, dtype: int64

---customer_occupation---
<StringArray>
['Doctor', 'Student', 'Retired', 'Engineer']
Length: 4, dtype: str
customer_occupation
Student     657
Doctor      631
Engineer    625
Retired     599
Name: count, dtype: int64


In [21]:
inconsistent = df.groupby('account_id').agg(
    age_variations        = ('customer_age', 'nunique'),
    occupation_variations = ('customer_occupation', 'nunique')
)

problems = inconsistent[
    (inconsistent['age_variations'] > 1) |
    (inconsistent['occupation_variations'] > 1)
]

print(f"Accounts with inconsistent age or occupation: {len(problems)}")
print(problems)

Accounts with inconsistent age or occupation: 471
            age_variations  occupation_variations
account_id                                       
AC00001                  2                      2
AC00002                  7                      3
AC00003                  5                      3
AC00004                  9                      4
AC00005                  9                      4
...                    ...                    ...
AC00496                  3                      3
AC00497                  5                      3
AC00498                  8                      3
AC00499                  7                      4
AC00500                  4                      2

[471 rows x 2 columns]
